####Ingestion del archivo "movie_cast.json"

In [0]:
dbutils.widgets.text("p_file_date", "")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/common_functions"

###Paso 1 - Leer el archivo JSON usando DataFrameReader de Spark

In [0]:
from pyspark.sql.types import StructField, StructType, IntegerType, StringType

In [0]:
movies_cast_schema = StructType(fields=[
    StructField("movieId", IntegerType(), True),
    StructField("personId", IntegerType(), True),
    StructField("characterName", StringType(), True),
    StructField("genderId", IntegerType(), True),
    StructField("castOrder", IntegerType(), True),
])

In [0]:
movies_casts_df = spark.read \
                .schema(movies_cast_schema) \
                .option("multiline", True) \
                .json(f"abfss://bronze@moviehistory8.dfs.core.windows.net/{v_file_date}/movie_cast.json")

In [0]:
display(movies_casts_df)

movieId,personId,characterName,genderId,castOrder
13673,7427,Merry,1,13
13673,15031,Aubie,0,14
13673,1599175,Officer Staff,2,15
13673,95875,Husband,2,16
13836,18918,Jack Bruno,2,0
13836,1285,Sara,1,1
13836,23498,Seth,2,2
13836,17832,Dr. Alex Friedman,1,3
13836,59238,Pope,2,4
13836,53261,Tina,1,5


###Paso 2 - Renombrar las columnas y añadir nuevas columnas

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
movies_casts_with_columns_df = movies_casts_df \
                                .withColumnsRenamed({"movieId": "movie_id",
                                                    "personId": "person_id",
                                                    "characterName": "character_name"}) \
                                .withColumn("ingestion_date", current_timestamp()) \
                                .withColumn("enviroment", lit("Production")) \
                                .withColumn("file_date", lit(v_file_date))

###Paso 3 -  Eliminar las columnas no deseadas

In [0]:
from pyspark.sql.functions import col

In [0]:
movies_casts_final_df = movies_casts_with_columns_df.drop("genderId", "castOrder")

In [0]:
display(movies_casts_final_df)

movie_id,person_id,character_name,ingestion_date,enviroment,file_date
13673,7427,Merry,2026-09-28T09:10:48.348402Z,Production,2024-12-30
13673,15031,Aubie,2026-09-28T09:10:48.348402Z,Production,2024-12-30
13673,1599175,Officer Staff,2026-09-28T09:10:48.348402Z,Production,2024-12-30
13673,95875,Husband,2026-09-28T09:10:48.348402Z,Production,2024-12-30
13836,18918,Jack Bruno,2026-09-28T09:10:48.348402Z,Production,2024-12-30
13836,1285,Sara,2026-09-28T09:10:48.348402Z,Production,2024-12-30
13836,23498,Seth,2026-09-28T09:10:48.348402Z,Production,2024-12-30
13836,17832,Dr. Alex Friedman,2026-09-28T09:10:48.348402Z,Production,2024-12-30
13836,59238,Pope,2026-09-28T09:10:48.348402Z,Production,2024-12-30
13836,53261,Tina,2026-09-28T09:10:48.348402Z,Production,2024-12-30


###Paso 4 - Escribir la salida en formato Parquet

In [0]:
#overwrite_partition("movie_silver", "movies_casts", "file_date", v_file_date)

In [0]:
#movies_casts_final_df.write.mode("overwrite").parquet("abfss://silver@moviehistory8.dfs.core.windows.net/movies_casts")

In [0]:
merge_condition = 'tgt.movie_id = src.movie_id AND tgt.person_id = src.person_id AND tgt.file_date = src.file_date'
merge_delta_lake(movies_casts_final_df, "movie_silver", "movies_casts", merge_condition, "file_date")

In [0]:
#movies_casts_final_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("movie_silver.movies_casts")

In [0]:
%sql
SELECT file_date, count(*)
FROM movie_silver.movies_casts
GROUP BY file_date;

file_date,count(1)
2024-12-23,15000
2024-12-16,30000
2024-12-30,5000


In [0]:
%sql
SELECT * FROM movie_silver.movies_casts;

movie_id,person_id,character_name,ingestion_date,enviroment,file_date
9448,43426,Gump,2026-09-28T09:10:27.771772Z,Production,2024-12-23
5876,11155,David Drayton,2026-09-28T09:10:27.771772Z,Production,2024-12-23
5876,4726,Mrs. Carmody,2026-09-28T09:10:27.771772Z,Production,2024-12-23
5876,8332,Amanda,2026-09-28T09:10:27.771772Z,Production,2024-12-23
5876,6861,Norton,2026-09-28T09:10:27.771772Z,Production,2024-12-23
5876,13014,Ollie,2026-09-28T09:10:27.771772Z,Production,2024-12-23
5876,6573,Jim,2026-09-28T09:10:27.771772Z,Production,2024-12-23
5876,12645,Dan Miller,2026-09-28T09:10:27.771772Z,Production,2024-12-23
5876,36926,Irene,2026-09-28T09:10:27.771772Z,Production,2024-12-23
5876,18052,Billy Drayton,2026-09-28T09:10:27.771772Z,Production,2024-12-23


In [0]:
#display(spark.read.parquet("abfss://silver@moviehistory8.dfs.core.windows.net/movies_casts"))

In [0]:
dbutils.notebook.exit("Success")